In [1]:
# Cell 1: import library
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

import time, json, warnings

In [2]:
df = pd.read_parquet("../data/processed/features.parquet")
df.head()

,d_num,date,sales,sell_price,price_diff_7,price_ratio_28,price_ratio_max,price_ratio_182,lag_28,lag_35,...,is_christmas,has_event,snap,item_id,dept_id,cat_id,event_name_1,event_type_1,event_name_2,event_type_2
0,1,2011-01-29,3,2.0,NaN,NaN,1.0,NaN,NaN,NaN,...,0,0,0,FOODS_1_001,FOODS_1,FOODS,none,none,none,none
1,2,2011-01-30,0,2.0,NaN,NaN,1.0,NaN,NaN,NaN,...,0,0,0,FOODS_1_001,FOODS_1,FOODS,none,none,none,none
2,3,2011-01-31,0,2.0,NaN,NaN,1.0,NaN,NaN,NaN,...,0,0,0,FOODS_1_001,FOODS_1,FOODS,none,none,none,none
3,4,2011-02-01,1,2.0,NaN,NaN,1.0,NaN,NaN,NaN,...,0,0,1,FOODS_1_001,FOODS_1,FOODS,none,none,none,none
4,5,2011-02-02,4,2.0,NaN,NaN,1.0,NaN,NaN,NaN,...,0,0,1,FOODS_1_001,FOODS_1,FOODS,none,none,none,none


In [3]:
# Cell 2: setup feature lists
meta = json.loads(Path("../data/processed/features_meta.json").read_text())
NUM, CAT, TARGET = meta["NUM"], meta["CAT"], meta["TARGET"]
H, LAST_DAY = meta["H"], meta["LAST_DAY"]
FEATURES = NUM + CAT
TEST_START = LAST_DAY - H + 1

df["sales"] = df["sales"].astype("float32")

Y = df.pivot(index="item_id", columns="d_num", values="sales").reindex(columns=range(1, LAST_DAY + 1))
Y_np = Y.to_numpy()
date_of = df.drop_duplicates("d_num").set_index("d_num")["date"]


# folds
rows = []
for i in (1, 2, 3):
    val_end = TEST_START - 1 -(3 -i) * H
    val_start = val_end - H + 1
    rows.append({"fold": i, "train_end": val_start - 1, "val_start": val_start, "val_end": val_end})
folds = pd.DataFrame(rows)

# metrics
def _valid(y, p):
    m = ~np.isnan(y) & ~np.isnan(p)
    return y[m], p[m]

def mae(y, p):
    y, p = _valid(y, p)
    return float(np.abs(y - p).mean())

def rmse(y, p):
    y, p = _valid(y, p)
    return float(np.sqrt(((y - p) ** 2).mean()))

def wape(y, p):
    y, p = _valid(y, p)
    return float(np.abs(y - p).sum() / y.sum())

def mase_scale(Y_train, m=7):
    d = np.abs(Y_train[:, m:] - Y_train[:, :-m])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(d, axis=1)

def mase(y, p, scale):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        ratio = np.nanmean(np.abs(y - p), axis=1) / scale
    ok = np.isfinite(ratio) & (scale > 0)
    return float(ratio[ok].mean()), int((~ok).sum())

def score(Y_val, P, Y_train):
    m, skipped = mase(Y_val, P, mase_scale(Y_train))
    return {"MAE": mae(Y_val, P), "RMSE": rmse(Y_val, P),
            "WAPE": wape(Y_val, P), "MASE": m, "mase_items_skipped": skipped}

# moving average baseline, for the re-check
def moving_average(Y_train, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        ma = np.nanmean(Y_train[:, -w:], axis=1)
    return np.nan_to_num(np.repeat(ma[:, None], h, axis=1), nan=0.0)

def backtest(forecaster, name, Y_all, folds, h=H, **kw):
    out = []
    for _, f in folds.iterrows():
        te, vs, ve = int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        assert vs == te + 1 and ve - vs + 1 == h
        Y_train, Y_val = Y_all[:, :te], Y_all[:, vs - 1: ve]
        P = forecaster(Y_train, h=h, **kw)
        out.append({"model": name, "fold": int(f["fold"]), **score(Y_val, P, Y_train)})
    return pd.DataFrame(out)

#  re-check
print("lightgbm version:", lgb.__version__)
print("table shape :", df.shape)                                        # expect (4788267, 36)
print("features    :", len(NUM), "numeric +", len(CAT), "categorical")  # expect 26 + 7
print("Y shape     :", Y.shape, "| NaN cells:", int(np.isnan(Y_np).sum()))   # expect (3049, 1941) | 1129842
print("total units :", int(np.nansum(Y_np)))                            # expect 7832248
chk = backtest(moving_average, "moving average (28d)", Y_np, folds)
print("moving average WAPE:", round(chk["WAPE"].mean(), 3))             # expect 0.748
display(folds)

lightgbm version: 4.7.0
table shape : (4788267, 36)
features    : 26 numeric + 7 categorical
Y shape     : (3049, 1941) | NaN cells: 1129842
total units : 7832248
moving average WAPE: 0.748


,fold,train_end,val_start,val_end
0,1,1829,1830,1857
1,2,1857,1858,1885
2,3,1885,1886,1913


In [4]:
# Cell 3: Training function, with a time test on fold 1
CAT_A = [c for c in CAT if c != "item_id"]
FEATS_A = NUM + CAT_A
pos_map = pd.Series(np.arange(len(Y)), index = Y.index.astype(str))

PARAMS = dict(objective = "poisson", learning_rate=0.1, num_leaves=63,
             min_data_in_leaf=100, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1,
              seed=42, verbose=-1, force_row_wise=True)

def fit_predict(fold, feats, cats, params, rounds, train_start=1):
    te, vs, ve = int(fold["train_end"]), int(fold["val_start"]), int(fold["val_end"])
    assert TARGET not in feats and "d_num" not in feats, "forbidden column in features"
    train = df[(df["d_num"] >= train_start) & (df["d_num"] <= te)]
    val = df[(df["d_num"] >= vs) & (df["d_num"] <= ve)]
    assert train["d_num"].max() == te and val["d_num"].min() == vs and val["d_num"].max() == ve

    dtrain = lgb.Dataset(train[feats], label=train[TARGET],
                         categorical_feature=cats, free_raw_data=True)
    booster = lgb.train(params, dtrain, num_boost_round=rounds)

    pred = booster.predict(val[feats])
    P = np.full((len(Y), H), np.nan, dtype="float32")
    P[pos_map[val["item_id"].astype(str)].to_numpy(), val["d_num"].to_numpy() - vs] = pred
    Y_train, Y_val = Y_np[:, :te], Y_np[:, vs - 1: ve]
    return booster, P, Y_val, Y_train, len(train)


# time test: fold 1, 100 rounds

f1 = folds.iloc[0]
t0 = time.time()
booster, P, Y_val, Y_train, n_train = fit_predict(f1, FEATS_A, CAT_A, PARAMS, rounds=100)
sec = time.time() - t0
print(f"fold 1, 100 rounds: {sec:.0f}s | training rows: {n_train}")
print(f"estimate: 400 rounds x 3 folds = about {sec * 4 * 3 / 60:.0f} min")

print("\ncells with an actual but no prediction:", int((~np.isnan(Y_val) & np.isnan(P)).sum()))   # expect 0
print("cells with a prediction but no actual:", int((np.isnan(Y_val) & ~np.isnan(P)).sum()))      # expect 0
print("lowest prediction:", round(float(np.nanmin(P)), 4), "(Poisson should not go below 0)")

print("\nLightGBM, fold 1       :", {k: round(v, 3) for k, v in score(Y_val, P, Y_train).items()})
ma = moving_average(Y_train)
print("moving average, fold 1 :", {k: round(v, 3) for k, v in score(Y_val, ma, Y_train).items()})

imp = pd.Series(booster.feature_importance("gain"), index=FEATS_A).sort_values(ascending=False)
print("\ntop 10 features by gain (%):")
print((imp / imp.sum() * 100).round(1).head(10).to_string())

fold 1, 100 rounds: 17s | training rows: 4446791
estimate: 400 rounds x 3 folds = about 3 min

cells with an actual but no prediction: 0
cells with a prediction but no actual: 0
lowest prediction: 0.1176 (Poisson should not go below 0)

LightGBM, fold 1       : {'MAE': 1.116, 'RMSE': 2.255, 'WAPE': 0.775, 'MASE': 0.981, 'mase_items_skipped': 1}
moving average, fold 1 : {'MAE': 1.075, 'RMSE': 2.234, 'WAPE': 0.747, 'MASE': 0.871, 'mase_items_skipped': 1}

top 10 features by gain (%):
roll_mean_14    42.4
roll_mean_28    37.4
roll_mean_56     5.9
roll_mean_7      3.0
lag_28           1.8
dow              1.5
sell_price       1.4
dept_id          1.2
roll_std_28      0.8
age              0.6


In [6]:
# Cell 3b: Error by horizon week
rows = []
for k in range(4):
    sl = slice(7 * k, 7 * k + 7)
    rows.append({
        "horizon days": f"{7*k}-{7*k+7}",
        "feature age (days)": f"{27-7*k} to {21-7*k}",
        "LightGBM WAPE": wape(Y_val[:, sl], P[:, sl]),
        "moving avg WAPE": wape(Y_val[:, sl], ma[:, sl]),
    })

by_week = pd.DataFrame(rows)
by_week["gap (LGB - MA)"] = by_week["LightGBM WAPE"] - by_week["moving avg WAPE"]
display(by_week.round(3))

,horizon days,feature age (days),LightGBM WAPE,moving avg WAPE,gap (LGB - MA)
0,0-7,27 to 21,0.770,0.714,0.055
1,7-14,20 to 14,0.798,0.759,0.039
2,14-21,13 to 7,0.759,0.736,0.023
3,21-28,6 to 0,0.776,0.780,-0.004


In [8]:
# Cell 4: the block models
bm = json.loads(Path("../data/processed/features_blocks_meta.json").read_text())
NUM_B = bm["NUM"]
CAT_B = [c for  c in bm["CAT"] if c != "item_id"]
FEATS_B = NUM_B + CAT_B
BLOCKS = {int(k): v for k, v in bm["blocks"].items()}

PARAMS = dict(objective="poisson", learning_rate=0.1, num_leaves=63, min_data_in_leaf=100,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              seed=42, verbose=-1, force_row_wise=True)
ROUNDS = 300
pos_map = pd.Series(np.arange(len(Y)), index=Y.index.astype(str))

# del df to frees about 500 MB if your laptop gets slow (the loop below does not use df)

P_blocks = {int(f["fold"]): np.full((len(Y), H), np.nan, dtype="float32") for _, f in folds.iterrows()}

t_all = time.time()
for b, s in BLOCKS.items():
    t0 = time.time()
    tbl = pd.read_parquet(f"../data/processed/features_s{s}.parquet",
                          columns=["d_num", "sales", "item_id"] + FEATS_B)

    for _, f in folds.iterrows():
        te, vs = int(f["train_end"]), int(f["val_start"])
        d0, d1 = vs + 7 * (b - 1), vs + 7 * b - 1             # horizon days of this block
        assert d1 - s <= te, "features would reach beyond the cutoff"

        train = tbl[tbl["d_num"] <= te]
        val = tbl[(tbl["d_num"] >= d0) & (tbl["d_num"] <= d1)]

        dtrain = lgb.Dataset(train[FEATS_B], label=train["sales"],
                             categorical_feature=CAT_B, free_raw_data=True)
        booster = lgb.train(PARAMS, dtrain, num_boost_round=ROUNDS)
        pred = booster.predict(val[FEATS_B])

        rows_i = pos_map[val["item_id"].astype(str)].to_numpy()
        P_blocks[int(f["fold"])][rows_i, val["d_num"].to_numpy() - vs] = pred
        del dtrain, booster
    del tbl
    print(f"block {b} (shift {s}) done in {time.time() - t0:.0f}s")
print(f"total: {(time.time() - t_all) / 60:.1f} min")

# scoring
rows, Yv, MAf = [], {}, {}
for _, f in folds.iterrows():
    fo, te, vs, ve = int(f["fold"]), int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
    Y_train, Y_val = Y_np[:, :te], Y_np[:, vs - 1: ve]
    P = P_blocks[fo]
    assert int((~np.isnan(Y_val) & np.isnan(P)).sum()) == 0, "actual without prediction"
    assert int((np.isnan(Y_val) & ~np.isnan(P)).sum()) == 0, "prediction without actual"
    MA = moving_average(Y_train)
    Yv[fo], MAf[fo] = Y_val, MA
    rows.append({"model": "LightGBM blocks (poisson, 300 rounds)", "fold": fo, **score(Y_val, P, Y_train)})
    rows.append({"model": "moving average (28d)", "fold": fo, **score(Y_val, MA, Y_train)})

res_blocks = pd.DataFrame(rows)
display(res_blocks.groupby("model")[["MAE", "RMSE", "WAPE", "MASE"]].mean().round(3))
print("WAPE per fold:")
display(res_blocks.pivot(index="model", columns="fold", values="WAPE").round(3))

# error by horizon week (mean over the 3 folds)

wk = []
for k in range(4):
    sl = slice(7 * k, 7 * k + 7)
    lg = np.mean([wape(Yv[fo][:, sl], P_blocks[fo][:, sl]) for fo in Yv])
    ma_ = np.mean([wape(Yv[fo][:, sl], MAf[fo][:, sl]) for fo in Yv])
    wk.append({"horizon days": f"{7*k+1}-{7*k+7}", "LightGBM WAPE": lg,
               "moving avg WAPE": ma_, "gap (LGB - MA)": lg - ma_})
display(pd.DataFrame(wk).round(3))

results_log = [res_blocks.assign(run="blocks, poisson, 300 rounds, no item_id")]


block 1 (shift 7) done in 104s
block 2 (shift 14) done in 113s
block 3 (shift 21) done in 116s
block 4 (shift 28) done in 110s
total: 7.4 min


,MAE,RMSE,WAPE,MASE
model,,,,
"LightGBM blocks (poisson, 300 rounds)",1.093,2.178,0.744,0.944
moving average (28d),1.099,2.257,0.748,0.895


WAPE per fold:


fold,1,2,3
model,,,
"LightGBM blocks (poisson, 300 rounds)",0.751,0.741,0.741
moving average (28d),0.747,0.753,0.745


,horizon days,LightGBM WAPE,moving avg WAPE,gap (LGB - MA)
0,1-7,0.717,0.723,-0.006
1,8-14,0.748,0.742,0.006
2,15-21,0.753,0.760,-0.007
3,22-28,0.760,0.769,-0.009


In [9]:
# Cell 5: Error by level (item, category, store), and a check on the MASE hypothesis

def seasonal_naive(Y_train, h=H, m=7):
    fc = np.tile(Y_train[:, -m:], (1, int(np.ceil(h / m))))[:, :h]
    return np.nan_to_num(fc, nan=0.0)

SN = {int(f["fold"]): seasonal_naive(Y_np[:, : int(f["train_end"])]) for _, f in folds.iterrows()}

cat_arr = (df.drop_duplicates("item_id").set_index("item_id")["cat_id"]
           .reindex(Y.index).astype(str).to_numpy())

cats = sorted(set(cat_arr))

def wape_arr(a, f):
    return float(np.abs(a - f).sum() / a.sum())

def levels(Pd):
    out = {"item-day": [], "category-day": [], "store-day": [], **{f"item-day {c}": [] for c in cats}}
    for fo, Yval in Yv.items():
        P = np.where(np.isnan(Yval), np.nan, Pd[fo])
        out["item-day"].append(wape(Yval, P))
        a_c = np.vstack([np.nansum(Yval[cat_arr == c], axis=0) for c in cats])
        p_c = np.vstack([np.nansum(P[cat_arr == c], axis=0) for c in cats])
        out["category-day"].append(wape_arr(a_c, p_c))
        out["store-day"].append(wape_arr(np.nansum(Yval, axis=0), np.nansum(P, axis=0)))
        for c in cats:
            out[f"item-day {c}"].append(wape(Yval[cat_arr == c], P[cat_arr == c]))
    return {k: float(np.mean(v)) for k, v in out.items()}

lv = pd.DataFrame({"LightGBM blocks": levels(P_blocks),
                   "moving average": levels(MAf),
                   "seasonal naive": levels(SN)}).T
display(lv.round(3))


print("\nshare of moving-average forecasts that are exactly 0:",
      round(float(np.mean([np.mean(MAf[fo] == 0) for fo in MAf])), 3))
print("lowest LightGBM prediction:", round(float(min(np.nanmin(P_blocks[fo]) for fo in P_blocks)), 4))


,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD
LightGBM blocks,0.744,0.072,0.056,0.682,0.980,0.804
moving average,0.748,0.171,0.164,0.689,0.971,0.806
seasonal naive,0.877,0.119,0.102,0.805,1.162,0.941



share of moving-average forecasts that are exactly 0: 0.097
lowest LightGBM prediction: 0.0369


In [14]:
# Cell 6: Turn the block training into a function

def run_blocks(name, params, rounds=300, use_item_id=False):
    cats = bm["CAT"] if use_item_id else [c for c in bm["CAT"] if c != "item_id"]
    feats = bm["NUM"] + cats
    cols = ["d_num", "sales"] + list(dict.fromkeys(["item_id"] + feats))
    P_run = {int(f["fold"]): np.full((len(Y), H), np.nan, dtype="float32") for _, f in folds.iterrows()}

    t_all = time.time()
    for b, s in BLOCKS.items():
        tbl = pd.read_parquet(f"../data/processed/features_s{s}.parquet", columns=cols)
        for _, f in folds.iterrows():
            te, vs = int(f["train_end"]), int(f["val_start"])
            d0, d1 = vs + 7 * (b - 1), vs + 7 * b - 1
            assert d1 - s <= te, "features would reach beyond the cutoff"
            train = tbl[tbl["d_num"] <= te]
            val = tbl[(tbl["d_num"] >= d0) & (tbl["d_num"] <= d1)]
            dtrain = lgb.Dataset(train[feats], label=train["sales"],
                                 categorical_feature = cats, free_raw_data=True)
            booster = lgb.train(params, dtrain, num_boost_round=rounds)
            pred = booster.predict(val[feats])
            rows_i = pos_map[val["item_id"].astype(str)].to_numpy()
            P_run[int(f["fold"])][rows_i, val["d_num"].to_numpy() - vs] = pred
            del dtrain, booster
        del tbl
    rows = []
    for _, f in folds.iterrows():
        fo, te, vs, ve = int(f["fold"]), int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        Y_train, Y_val = Y_np[:, :te], Y_np[:, vs - 1: ve]
        P = P_run[fo]
        assert int((~np.isnan(Y_val) & np.isnan(P)).sum()) == 0
        assert int((np.isnan(Y_val) & ~np.isnan(P)).sum()) == 0
        rows.append({"model": name, "fold": fo, **score(Y_val, P, Y_train)})
    res = pd.DataFrame(rows)
    print(f"{name}: WAPE {res['WAPE'].mean():.3f} | MASE {res['MASE'].mean():.3f} "
          f"| {(time.time() - t_all) / 60:.1f} min")
    return P_run, res

runs = {"poisson (300 rounds)": P_blocks}

def store_by_fold(P):
    return [round(wape_arr(np.nansum(Yv[fo], axis=0),
                           np.nansum(np.where(np.isnan(Yv[fo]), np.nan, P[fo]), axis=0)), 3)
            for fo in Yv]

def compare():
    lv = pd.DataFrame({k: levels(P) for k, P in runs.items()}).T
    lv["store-day by fold"] = [store_by_fold(P) for P in runs.values()]
    display(lv.round(3))
    display(pd.concat(results_log).groupby("model", sort=False)[["MAE", "RMSE", "WAPE", "MASE"]].mean().round(3))

compare()


,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD,store-day by fold
poisson (300 rounds),0.744,0.072,0.056,0.682,0.98,0.804,"[0.055, 0.062, 0.052]"


,MAE,RMSE,WAPE,MASE
model,,,,
"LightGBM blocks (poisson, 300 rounds)",1.093,2.178,0.744,0.944
moving average (28d),1.099,2.257,0.748,0.895


In [15]:
# Cell 7: Experiment 1, the Tweedie objective

P_tw, res_tw = run_blocks("tweedie 1.2 (300 rounds)",
                          {**PARAMS, "objective": "tweedie", "tweedie_variance_power": 1.2})
runs["tweedie 1.2"] = P_tw
results_log.append(res_tw)
compare()

tweedie 1.2 (300 rounds): WAPE 0.743 | MASE 0.944 | 7.2 min


,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD,store-day by fold
poisson (300 rounds),0.744,0.072,0.056,0.682,0.980,0.804,"[0.055, 0.062, 0.052]"
tweedie 1.2,0.743,0.074,0.058,0.680,0.979,0.805,"[0.059, 0.06, 0.056]"


,MAE,RMSE,WAPE,MASE
model,,,,
"LightGBM blocks (poisson, 300 rounds)",1.093,2.178,0.744,0.944
moving average (28d),1.099,2.257,0.748,0.895
tweedie 1.2 (300 rounds),1.091,2.163,0.743,0.944


In [16]:
# Cell 8: Experiment 2: the L1 objective (a direct test of the MASE hypothesis)
def bias(P):
    return float(np.mean([np.nansum(np.where(np.isnan(Yv[fo]), np.nan, P[fo])) / np.nansum(Yv[fo])
                          for fo in Yv]))

P_l1, res_l1 = run_blocks("l1 (300 rounds)", {**PARAMS, "objective": "regression_l1"})
runs["l1"] = P_l1
results_log.append(res_l1)
compare()

print("bias (total forecast / total actual):", {k: round(bias(P), 3) for k, P in runs.items()})
vals = np.concatenate([P_l1[fo][~np.isnan(P_l1[fo])] for fo in P_l1])
print("share of L1 predictions below 0.05:", round(float((vals < 0.05).mean()), 3))


l1 (300 rounds): WAPE 0.674 | MASE 0.757 | 13.0 min


,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD,store-day by fold
poisson (300 rounds),0.744,0.072,0.056,0.682,0.980,0.804,"[0.055, 0.062, 0.052]"
tweedie 1.2,0.743,0.074,0.058,0.680,0.979,0.805,"[0.059, 0.06, 0.056]"
l1,0.674,0.306,0.306,0.634,0.841,0.707,"[0.324, 0.302, 0.292]"


,MAE,RMSE,WAPE,MASE
model,,,,
"LightGBM blocks (poisson, 300 rounds)",1.093,2.178,0.744,0.944
moving average (28d),1.099,2.257,0.748,0.895
tweedie 1.2 (300 rounds),1.091,2.163,0.743,0.944
l1 (300 rounds),0.990,2.226,0.674,0.757


bias (total forecast / total actual): {'poisson (300 rounds)': 1.005, 'tweedie 1.2': 1.008, 'l1': 0.694}
share of L1 predictions below 0.05: 0.516


In [17]:
# Cell 9: moving median baseline
def moving_median(Y_train, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        md = np.nanmedian(Y_train[:, -w:], axis=1)
    return np.nan_to_num(np.repeat(md[:, None], h, axis=1), nan=0.0)

MED = {int(f["fold"]): moving_median(Y_np[:, : int(f["train_end"])]) for _, f in folds.iterrows()}

rows = []
for _, f in folds.iterrows():
    fo, te, vs, ve = int(f["fold"]), int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
    rows.append({"model": "moving median (28d)", "fold": fo,
                 **score(Y_np[:, vs - 1: ve], MED[fo], Y_np[:, :te])})

results_log.append(pd.DataFrame(rows))
runs["moving median"] = MED

compare()
print("bias:", {k: round(bias(P), 3) for k, P in runs.items()})
print("share of moving-median forecasts that are exactly 0:",
      round(float(np.mean([np.mean(MED[fo] == 0) for fo in MED])), 3))

,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD,store-day by fold
poisson (300 rounds),0.744,0.072,0.056,0.682,0.980,0.804,"[0.055, 0.062, 0.052]"
tweedie 1.2,0.743,0.074,0.058,0.680,0.979,0.805,"[0.059, 0.06, 0.056]"
l1,0.674,0.306,0.306,0.634,0.841,0.707,"[0.324, 0.302, 0.292]"
moving median,0.698,0.276,0.274,0.660,0.847,0.731,"[0.286, 0.27, 0.265]"


,MAE,RMSE,WAPE,MASE
model,,,,
"LightGBM blocks (poisson, 300 rounds)",1.093,2.178,0.744,0.944
moving average (28d),1.099,2.257,0.748,0.895
tweedie 1.2 (300 rounds),1.091,2.163,0.743,0.944
l1 (300 rounds),0.990,2.226,0.674,0.757
moving median (28d),1.024,2.360,0.698,0.771


bias: {'poisson (300 rounds)': 1.005, 'tweedie 1.2': 1.008, 'l1': 0.694, 'moving median': 0.726}
share of moving-median forecasts that are exactly 0: 0.56


In [18]:
# Cell 10: Are the gains consistent? (no retraining)
allr = pd.concat(results_log)
print("WAPE per fold:")
display(allr.pivot_table(index="model", columns="fold", values="WAPE", sort=False).round(3))
print("MASE per fold:")
display(allr.pivot_table(index="model", columns="fold", values="MASE", sort=False).round(3))

def week_wape(Pd, k):
    sl = slice(7 * k, 7 * k + 7)
    return np.mean([wape(Yv[fo][:, sl], Pd[fo][:, sl]) for fo in Yv])

rows = []
for k in range(4):
    rows.append({"horizon days": f"{7*k+1}-{7*k+7}",
                 "poisson - moving avg": week_wape(P_blocks, k) - week_wape(MAf, k),
                 "l1 - moving median": week_wape(P_l1, k) - week_wape(MED, k)})
print("gap by horizon week (negative = LightGBM better):")
display(pd.DataFrame(rows).round(3))

WAPE per fold:


fold,1,2,3
model,,,
"LightGBM blocks (poisson, 300 rounds)",0.751,0.741,0.741
moving average (28d),0.747,0.753,0.745
tweedie 1.2 (300 rounds),0.751,0.738,0.739
l1 (300 rounds),0.680,0.674,0.669
moving median (28d),0.698,0.696,0.700


MASE per fold:


fold,1,2,3
model,,,
"LightGBM blocks (poisson, 300 rounds)",0.941,0.938,0.954
moving average (28d),0.871,0.902,0.913
tweedie 1.2 (300 rounds),0.941,0.937,0.955
l1 (300 rounds),0.742,0.760,0.769
moving median (28d),0.751,0.773,0.789


gap by horizon week (negative = LightGBM better):


,horizon days,poisson - moving avg,l1 - moving median
0,1-7,-0.006,-0.030
1,8-14,0.006,-0.017
2,15-21,-0.007,-0.025
3,22-28,-0.009,-0.021


In [19]:
# Cell 11:  Experiment 3: item_id as a feature (with Poisson)
P_id, res_id = run_blocks("poisson + item_id (300 rounds)", PARAMS, use_item_id=True)
runs["poisson + item_id"] = P_id
results_log.append(res_id)
compare()
print("bias:", {k: round(bias(P), 3) for k, P in runs.items()})

poisson + item_id (300 rounds): WAPE 0.739 | MASE 0.899 | 9.1 min


,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD,store-day by fold
poisson (300 rounds),0.744,0.072,0.056,0.682,0.980,0.804,"[0.055, 0.062, 0.052]"
tweedie 1.2,0.743,0.074,0.058,0.680,0.979,0.805,"[0.059, 0.06, 0.056]"
l1,0.674,0.306,0.306,0.634,0.841,0.707,"[0.324, 0.302, 0.292]"
moving median,0.698,0.276,0.274,0.660,0.847,0.731,"[0.286, 0.27, 0.265]"
poisson + item_id,0.739,0.067,0.052,0.678,0.974,0.796,"[0.054, 0.059, 0.045]"


,MAE,RMSE,WAPE,MASE
model,,,,
"LightGBM blocks (poisson, 300 rounds)",1.093,2.178,0.744,0.944
moving average (28d),1.099,2.257,0.748,0.895
tweedie 1.2 (300 rounds),1.091,2.163,0.743,0.944
l1 (300 rounds),0.990,2.226,0.674,0.757
moving median (28d),1.024,2.360,0.698,0.771
poisson + item_id (300 rounds),1.084,2.181,0.739,0.899


bias: {'poisson (300 rounds)': 1.005, 'tweedie 1.2': 1.008, 'l1': 0.694, 'moving median': 0.726, 'poisson + item_id': 0.997}


In [21]:
# Cell 12: Experiment 4: how many rounds? (no extra cost per round count)
def score_run(P_run):
    rows = []
    for _, f in folds.iterrows():
        fo, te, vs, ve = int(f["fold"]), int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        rows.append(score(Y_np[:, vs - 1: ve], P_run[fo], Y_np[:, :te]))
    return pd.DataFrame(rows)[["WAPE", "MASE"]].mean()

def run_blocks_multi(params, rounds_list, use_item_id=True):
    cats = bm["CAT"] if use_item_id else [c for c in bm["CAT"] if c != "item_id"]
    feats = bm["NUM"] + cats
    cols = ["d_num", "sales"] + list(dict.fromkeys(["item_id"] + feats))
    P_runs = {r: {int(f["fold"]): np.full((len(Y), H), np.nan, dtype="float32")
                  for _, f in folds.iterrows()} for r in rounds_list}
    t_all = time.time()
    for b, s in BLOCKS.items():
        tbl = pd.read_parquet(f"../data/processed/features_s{s}.parquet", columns=cols)
        for _, f in folds.iterrows():
            fo, te, vs = int(f["fold"]), int(f["train_end"]), int(f["val_start"])
            d0, d1 = vs + 7 * (b - 1), vs + 7 * b - 1
            assert d1 - s <= te, "features would reach beyond the cutoff"
            train = tbl[tbl["d_num"] <= te]
            val = tbl[(tbl["d_num"] >= d0) & (tbl["d_num"] <= d1)]
            dtrain = lgb.Dataset(train[feats], label=train["sales"],
                                 categorical_feature=cats, free_raw_data=True)
            booster = lgb.train(params, dtrain, num_boost_round=max(rounds_list))
            rows_i = pos_map[val["item_id"].astype(str)].to_numpy()
            for r in rounds_list:
                P_runs[r][fo][rows_i, val["d_num"].to_numpy() - vs] = booster.predict(val[feats], num_iteration=r)
            del dtrain, booster
        del tbl
    print(f"done in {(time.time() - t_all) / 60:.1f} min")
    return P_runs

ROUNDS_LIST = [100, 200, 300, 450, 600]
P_rounds = run_blocks_multi(PARAMS, ROUNDS_LIST)

rows = []
for r in ROUNDS_LIST:
    lv = levels(P_rounds[r]); sc = score_run(P_rounds[r])
    rows.append({"rounds": r, "item-day WAPE": lv["item-day"], "MASE": sc["MASE"],
                 "category-day": lv["category-day"], "store-day": lv["store-day"],
                 "bias": bias(P_rounds[r])})
display(pd.DataFrame(rows).round(3))

done in 15.6 min


,rounds,item-day WAPE,MASE,category-day,store-day,bias
0,100,0.745,0.926,0.068,0.055,0.995
1,200,0.739,0.902,0.066,0.052,0.993
2,300,0.739,0.899,0.067,0.052,0.997
3,450,0.738,0.898,0.068,0.053,0.999
4,600,0.737,0.898,0.068,0.053,0.999


In [22]:
# Cell 13 : Experiment 5: num_leaves (31 against 127, with 63 as the reference)
leaf_runs = {63: P_rounds[300]}
for nl in (31, 127):
    print(f"--- num_leaves = {nl} ---")
    leaf_runs[nl] = run_blocks_multi({**PARAMS, "num_leaves": nl}, [300])[300]

rows = []
for nl in (31, 63, 127):
    P = leaf_runs[nl]
    lv, sc = levels(P), score_run(P)
    rows.append({"num_leaves": nl, "item-day WAPE": lv["item-day"], "MASE": sc["MASE"],
                 "category-day": lv["category-day"], "store-day": lv["store-day"],
                 "bias": bias(P), "store-day by fold": store_by_fold(P)})
display(pd.DataFrame(rows).round(3))

--- num_leaves = 31 ---
done in 10.9 min
--- num_leaves = 127 ---
done in 14.0 min


,num_leaves,item-day WAPE,MASE,category-day,store-day,bias,store-day by fold
0,31,0.740,0.902,0.067,0.052,0.995,"[0.053, 0.06, 0.044]"
1,63,0.739,0.899,0.067,0.052,0.997,"[0.054, 0.059, 0.045]"
2,127,0.737,0.898,0.068,0.053,0.993,"[0.055, 0.06, 0.044]"


In [23]:
# Cell 14 :Save the results table


Path("../reports").mkdir(exist_ok=True)
log = pd.concat(results_log, ignore_index=True)
log.to_csv("../reports/notebook06_results.csv", index=False)
print("saved:", Path("../reports/notebook06_results.csv").resolve())
print(log.groupby("model", sort=False)["WAPE"].mean().round(3).to_string())

saved: C:\Users\ASUS\Desktop\AI_Projects\Retail-demand-forecasting\reports\notebook06_results.csv
model
LightGBM blocks (poisson, 300 rounds)    0.744
moving average (28d)                     0.748
tweedie 1.2 (300 rounds)                 0.743
l1 (300 rounds)                          0.674
moving median (28d)                      0.698
poisson + item_id (300 rounds)           0.739


## Summary of Notebook 06: LightGBM

**Setup**
- Same 3 folds as before (4-week validation windows, 2016-02-01 to 2016-04-24); final test window untouched
- Features from Notebook 05; trained on all rows up to each fold's cutoff; no early stopping on the validation window
- No item_id unless stated

**Design finding: stale features**
- One model with sales features shifted by 28 days lost to the moving average on fold 1 (WAPE 0.775 vs 0.747); the gap shrank from +0.055 in days 1-7 to -0.004 in days 22-28 as features got fresher
- Fix: four models, one per horizon block (days 1-7, 8-14, 15-21, 22-28), with features shifted by 7, 14, 21, 28 days (multiples of 7 keep the weekday aligned); gap to the moving average then no longer depends on the horizon week

**Results (mean of 3 folds)**

| Model | item-day WAPE | MASE | MAE | RMSE | category-day WAPE | store-day WAPE | bias |
|---|---|---|---|---|---|---|---|
| moving average (28d) | 0.748 | 0.895 | 1.099 | 2.257 | 0.171 | 0.164 | n/a |
| moving median (28d) | 0.698 | 0.771 | 1.024 | 2.360 | 0.276 | 0.274 | 0.726 |
| LightGBM Poisson | 0.744 | 0.944 | 1.093 | 2.178 | 0.072 | 0.056 | 1.005 |
| LightGBM Tweedie 1.2 | 0.743 | 0.944 | 1.091 | 2.163 | 0.074 | 0.058 | 1.008 |
| LightGBM L1 (median) | 0.674 | 0.757 | 0.990 | 2.226 | 0.306 | 0.306 | 0.694 |
| **LightGBM Poisson + item_id** | 0.739 | 0.899 | 1.084 | 2.181 | 0.067 | 0.052 | 0.997 |

**Findings**
- Mean forecasts (Poisson): tie with the moving average per item and day, about 3 times better on category and store totals (0.052-0.067 vs 0.164-0.171)
- Median forecast (L1): 0.674 vs 0.698 for a moving median (better in all 3 folds and all 4 horizon weeks), but totals are under-forecast by about 30%; most of its gain over the moving average is the median effect (the moving median alone reaches 0.698)
- Objective (Poisson vs Tweedie): no difference beyond the fold spread
- item_id as a feature: small, consistent improvement (store-day better in all 3 folds), no sign of overfitting; MASE 0.944 -> 0.899
- Rounds (100/200/300/450/600): flat from 200 on (WAPE 0.739-0.737); num_leaves (31/63/127): WAPE 0.740/0.739/0.737; chosen: 300 rounds, 63 leaves
- WAPE and MASE rank mean and median forecasts differently because absolute error is minimised by the median (many items have a median of 0)

**Chosen model:** Poisson, item_id, 63 leaves, 300 rounds, four horizon-block models

**Limits (for the README)**
- Rounds and leaves were chosen on the same validation folds (slightly optimistic); the final test window is untouched
- Folds share most of their training data, so they are not independent tests
- Spring windows only; no holiday or summer fold
- Four models per fold (training about 9 minutes per full run)
- New items without history have no item_id effect and no lags in their first weeks
- Item-level gains over the best simple baseline are small; the clear gain is on category and store totals